# Retrieval Evaluation
This notebook evaluates Text Search, Vector Search, and Hybrid Search on the Wikipedia ML concepts dataset.

In [ ]:
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
import pandas as pd
import os

es = Elasticsearch(os.getenv('ELASTIC_URL', 'http://localhost:9200'))
model = SentenceTransformer('all-MiniLM-L6-v2')
INDEX_NAME = 'ml_concepts'

In [ ]:
def text_search(query, k=5):
    res = es.search(index=INDEX_NAME, body={'query': {'match': {'text': query}}, 'size': k})
    return [hit['_source']['id'] for hit in res['hits']['hits']]

def vector_search(query, k=5):
    vec = model.encode(query).tolist()
    res = es.search(index=INDEX_NAME, body={'knn': {'field': 'embedding', 'query_vector': vec, 'k': k, 'num_candidates': 50}, 'size': k})
    return [hit['_source']['id'] for hit in res['hits']['hits']]

def hybrid_search(query, k=5):
    vec = model.encode(query).tolist()
    res = es.search(index=INDEX_NAME, body={'knn': {'field': 'embedding', 'query_vector': vec, 'k': k, 'num_candidates': 50, 'boost': 0.5}, 'query': {'match': {'text': {'query': query, 'boost': 0.5}}}, 'size': k})
    return [hit['_source']['id'] for hit in res['hits']['hits']]


In [ ]:
# Generate mock ground truth data
ground_truth = [
    {'query': 'What is deep learning?', 'doc_id': '10'},
    {'query': 'How does a random forest work?', 'doc_id': '25'},
    {'query': 'Explain backpropagation in neural networks.', 'doc_id': '15'}
]
# Note: In a real scenario, you'd use LLMs to generate 100+ questions based on the chunks to evaluate.

In [ ]:
def hit_rate(results, ground_truth_id):
    return 1 if ground_truth_id in results else 0

def mrr(results, ground_truth_id):
    for i, doc_id in enumerate(results):
        if doc_id == ground_truth_id:
            return 1 / (i + 1)
    return 0

def evaluate_method(search_fn, ground_truth):
    total_hit = 0
    total_mrr = 0
    for item in ground_truth:
        results = search_fn(item['query'])
        total_hit += hit_rate(results, item['doc_id'])
        total_mrr += mrr(results, item['doc_id'])
    return total_hit / len(ground_truth), total_mrr / len(ground_truth)

In [ ]:
metrics = {
    'Text Search': evaluate_method(text_search, ground_truth),
    'Vector Search': evaluate_method(vector_search, ground_truth),
    'Hybrid Search': evaluate_method(hybrid_search, ground_truth)
}
df = pd.DataFrame(metrics, index=['Hit Rate', 'MRR']).T
print(df)